# Vision lab

Tuning aid for `docs/IMPROVEMENTS.md` section 2 on recorded runs (`kibitz run --run-dir DIR`).

- frames with the tracked corners drawn on
- warped boards (White's view, row 0 = rank 8) with the FEN labels
- per-square diff heatmaps between consecutive **commit** frames, true changed squares outlined (python-chess)
- changed vs unchanged diff distributions against the Rust threshold

Only "commit" frames are labels; "lost" / "missed" frames are shown but their `fen` may not match the board.
Without real footage, make synthetic runs with `python -m kibitz_tools.fake_run` (see `tools/README.md`).

In [ ]:
import sys
from collections import Counter
from pathlib import Path

import chess
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Polygon, Rectangle

sys.path.insert(0, str(Path.cwd().parent))  # tools/, when run from tools/lab
from kibitz_tools.analysis import pair_diffs, threshold
from kibitz_tools.board import SQ_PX, square_to_grid, warp_board
from kibitz_tools.labels import CLASSES, fen_labels
from kibitz_tools.rundir import load_run

# Run dirs to look at. Edit me.
RUNS = [Path("../data/synth/run0"), Path("../data/synth/run1")]
# InferConfig::default().min_change in crates/cg-vision/src/infer.rs
RUST_MIN_CHANGE = 0.2

runs = [load_run(p) for p in RUNS]
for r in runs:
    print(r.name, dict(Counter(f.what for f in r.frames)), "events:", len(r.events),
          "pgn moves:", len(list(r.pgn.mainline_moves())) if r.pgn else None)

## Frames with the corner overlay
Corners are `[a8, h8, h1, a1]`.

In [ ]:
def show_frame(f, ax):
    ax.imshow(f.rgb())
    if f.corners is not None:
        c = np.array(f.corners)
        ax.add_patch(Polygon(c, closed=True, fill=False, ec="lime", lw=1.5))
        for name, (x, y) in zip(["a8", "h8", "h1", "a1"], c):
            ax.text(x, y, name, color="yellow", fontsize=8)
    ax.set_title(f"{f.what} @ {f.ts_ms}", fontsize=8)
    ax.axis("off")

run = runs[0]
show = run.frames[:: max(1, len(run.frames) // 4)][:4]
fig, axes = plt.subplots(1, len(show), figsize=(4 * len(show), 3))
for f, ax in zip(show, np.atleast_1d(axes)):
    show_frame(f, ax)
plt.tight_layout()

## Warped boards with FEN labels

In [ ]:
def show_warped(f, ax, labels=True):
    w = warp_board(f.gray(), f.corners, SQ_PX)
    ax.imshow(w, cmap="gray", vmin=0, vmax=255)
    if labels and f.is_label:
        lab = fen_labels(f.fen)
        for s in range(64):
            if lab[s]:
                r, c = square_to_grid(s)
                ax.text(c * SQ_PX + 2, r * SQ_PX + 9, CLASSES[lab[s]], color="red", fontsize=7)
    ax.set_title(f"{f.what} {f.ts_ms}", fontsize=8)
    ax.axis("off")

commits = run.commits()
show = commits[:: max(1, len(commits) // 4)][:4]
fig, axes = plt.subplots(1, len(show), figsize=(3.2 * len(show), 3.2))
for f, ax in zip(show, np.atleast_1d(axes)):
    show_warped(f, ax)
plt.tight_layout()

## Per-square diff between consecutive commits
Heat = `square_diff` (the runtime's change score). Green boxes = squares that really changed.

In [ ]:
pairs = [p for r in runs for p in pair_diffs(r)]
print(len(pairs), "pairs")

def show_pair(p, ax, vmax=None):
    board = chess.Board(p.before.fen)
    grid = np.zeros((8, 8))
    for s in range(64):
        grid[square_to_grid(s)] = p.diff[s]
    im = ax.imshow(grid, cmap="magma", vmin=0, vmax=vmax)
    for s in np.nonzero(p.changed)[0]:
        r, c = square_to_grid(int(s))
        ax.add_patch(Rectangle((c - 0.5, r - 0.5), 1, 1, fill=False, ec="lime", lw=2))
    t = threshold(p.diff, RUST_MIN_CHANGE)
    ax.set_title(f"{board.san(p.move)}  t={t:.2f}", fontsize=8)
    ax.set_xticks(range(8), list("abcdefgh"), fontsize=6)
    ax.set_yticks(range(8), [str(8 - i) for i in range(8)], fontsize=6)
    return im

show = pairs[:8]
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for p, ax in zip(show, axes.ravel()):
    im = show_pair(p, ax, vmax=max(q.diff.max() for q in show))
fig.colorbar(im, ax=axes, shrink=0.6)

## Changed vs unchanged distributions
The threshold has to sit in the gap. Log x axis; the dashed line is the Rust `min_change`.

In [ ]:
changed = np.concatenate([p.diff[p.changed] for p in pairs])
unchanged = np.concatenate([p.diff[~p.changed] for p in pairs])
bins = np.logspace(np.log10(max(1e-3, min(changed.min(), unchanged.min()))), np.log10(max(changed.max(), unchanged.max())), 60)
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.hist(unchanged, bins=bins, alpha=0.6, label=f"unchanged ({unchanged.size})", log=True)
ax.hist(changed, bins=bins, alpha=0.6, label=f"changed ({changed.size})", log=True)
ax.axvline(RUST_MIN_CHANGE, ls="--", c="k", label=f"min_change {RUST_MIN_CHANGE}")
ax.set_xscale("log"); ax.set_xlabel("square_diff"); ax.legend()
print(f"max unchanged {unchanged.max():.3f}, min changed {changed.min():.3f}")

## Per-pair separation over the game
min changed − max unchanged per pair; below 0 means no threshold separates that pair. Watch for drops when hands, shadows or tall pieces (2.4) appear.

In [ ]:
sep = [p.diff[p.changed].min() - p.diff[~p.changed].max() for p in pairs]
fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(sep, ".-"); ax.axhline(0, c="r", lw=0.8)
ax.set_xlabel("pair"); ax.set_ylabel("separation")
worst = int(np.argmin(sep))
print("worst pair:", worst, pairs[worst].run, pairs[worst].move, f"{sep[worst]:.3f}")